# DistilBERT Text Classification Model

This notebook trains a DistilBERT model on the preprocessed text data with an 80-10-10 train-validation-test split.

## Section 1: Install Required Libraries

Install the necessary libraries for DistilBERT model training including transformers, torch, and scikit-learn.

In [ ]:
# Install required libraries
import subprocess
import sys

# Install transformers, torch, and scikit-learn
subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers", "torch", "scikit-learn", "numpy", "tqdm"])

## Section 2: Load and Prepare Data

Load the preprocessed data and verify that the 'other_posts' and 'label' columns are ready for model training.

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Load the preprocessed data
data = pd.read_csv('first_dataset.csv')

# Display basic information about the data
print("Dataset shape:", data.shape)
print("\nFirst few rows:")
print(data.head())
print("\nColumn names:")
print(data.columns.tolist())
print("\nData types:")
print(data.dtypes)
print("\nLabel distribution:")
print(data['label'].value_counts())

## Section 3: Split Data into Train, Validation, and Test Sets

Use scikit-learn's train_test_split to create 80% training, 10% validation, and 10% test sets.

In [ ]:
# Split data into train (80%) and temp (20%)
train_data, temp_data = train_test_split(
    data, 
    test_size=0.2, 
    random_state=42, 
    stratify=data['label']
)

# Split temp (20%) into validation (50% of 20% = 10%) and test (50% of 20% = 10%)
val_data, test_data = train_test_split(
    temp_data, 
    test_size=0.5, 
    random_state=42, 
    stratify=temp_data['label']
)

print(f"Training set size: {len(train_data)} ({len(train_data)/len(data)*100:.1f}%)")
print(f"Validation set size: {len(val_data)} ({len(val_data)/len(data)*100:.1f}%)")
print(f"Test set size: {len(test_data)} ({len(test_data)/len(data)*100:.1f}%)")

print("\nLabel distribution in each set:")
print("Training set:")
print(train_data['label'].value_counts())
print("\nValidation set:")
print(val_data['label'].value_counts())
print("\nTest set:")
print(test_data['label'].value_counts())

# Reset indices
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

## Section 4: Tokenize Text with DistilBERT

Use DistilBertTokenizer to tokenize the 'other_posts' text data with appropriate padding and truncation.

In [ ]:
# Initialize the DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Define tokenization function
def tokenize_function(texts, max_length=512):
    """
    Tokenize texts using DistilBERT tokenizer
    """
    encodings = tokenizer(
        texts.tolist(),
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    return encodings

# Tokenize train, validation, and test sets
print("Tokenizing training data...")
train_encodings = tokenize_function(train_data['other_posts'])

print("Tokenizing validation data...")
val_encodings = tokenize_function(val_data['other_posts'])

print("Tokenizing test data...")
test_encodings = tokenize_function(test_data['other_posts'])

print("\nTokenization complete!")
print(f"Train encodings shape: input_ids={train_encodings['input_ids'].shape}")
print(f"Validation encodings shape: input_ids={val_encodings['input_ids'].shape}")
print(f"Test encodings shape: input_ids={test_encodings['input_ids'].shape}")

## Section 5: Create PyTorch Datasets

Create custom PyTorch Dataset classes for train, validation, and test sets with tokenized inputs and labels.

In [ ]:
class TextClassificationDataset(Dataset):
    """
    Custom PyTorch Dataset for text classification with tokenized inputs
    """
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels.values, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

# Create datasets
train_dataset = TextClassificationDataset(train_encodings, train_data['label'])
val_dataset = TextClassificationDataset(val_encodings, val_data['label'])
test_dataset = TextClassificationDataset(test_encodings, test_data['label'])

print("Dataset Creation Complete!")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Create data loaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nData loaders created with batch size: {batch_size}")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## Section 6: Initialize DistilBERT Model

Load a pretrained DistilBERT model for sequence classification and configure training parameters.

In [ ]:
# Determine number of labels
num_labels = len(data['label'].unique())
print(f"Number of unique labels: {num_labels}")

# Load pretrained DistilBERT model for sequence classification
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

# Move model to device
model.to(device)

# Define optimizer and learning rate scheduler
optimizer = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_loader) * 3  # 3 epochs
scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, total_iters=total_steps)

print("\nModel Configuration:")
print(f"Model: DistilBERT for Sequence Classification")
print(f"Number of labels: {num_labels}")
print(f"Total training steps: {total_steps}")
print(f"Learning rate: 2e-5")
print(f"Device: {device}")

## Section 7: Train the Model

Train the DistilBERT model on the training set using a training loop, with validation on the validation set after each epoch.

In [ ]:
def train_epoch(model, train_loader, optimizer, scheduler, device):
    """
    Train the model for one epoch
    """
    model.train()
    total_loss = 0
    
    for batch in tqdm(train_loader, desc="Training"):
        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    return avg_loss

def evaluate(model, eval_loader, device):
    """
    Evaluate the model on validation or test set
    """
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []
    
    with torch.no_grad():
        for batch in tqdm(eval_loader, desc="Evaluating"):
            # Move batch to device
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop('labels')
            
            # Forward pass
            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits
            
            total_loss += loss.item()
            
            # Get predictions
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            predictions.extend(preds)
            true_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(eval_loader)
    accuracy = accuracy_score(true_labels, predictions)
    
    return avg_loss, accuracy, predictions, true_labels

# Training parameters
num_epochs = 3
best_val_accuracy = 0
patience = 1
patience_counter = 0

print("Starting model training...\n")

# Training loop
for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"{'='*50}")
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    print(f"Training Loss: {train_loss:.4f}")
    
    # Validate
    val_loss, val_accuracy, _, _ = evaluate(model, val_loader, device)
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")
    
    # Save best model
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save(model.state_dict(), 'best_distilbert_model.pt')
        print("Best model saved!")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch + 1} epochs")
            break

print("\nTraining completed!")

## Section 8: Evaluate on Validation and Test Sets

Evaluate model performance on both validation and test sets using accuracy, precision, recall, and F1-score metrics.

In [ ]:
# Load best model for final evaluation
model.load_state_dict(torch.load('best_distilbert_model.pt'))

print("="*60)
print("FINAL MODEL EVALUATION")
print("="*60)

# Evaluate on validation set
print("\n--- Validation Set Evaluation ---")
val_loss, val_accuracy, val_predictions, val_true_labels = evaluate(model, val_loader, device)
val_precision = precision_score(val_true_labels, val_predictions, average='weighted')
val_recall = recall_score(val_true_labels, val_predictions, average='weighted')
val_f1 = f1_score(val_true_labels, val_predictions, average='weighted')

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Precision: {val_precision:.4f}")
print(f"Validation Recall: {val_recall:.4f}")
print(f"Validation F1-Score: {val_f1:.4f}")

print("\nValidation Set Classification Report:")
print(classification_report(val_true_labels, val_predictions))

# Evaluate on test set
print("\n--- Test Set Evaluation ---")
test_loss, test_accuracy, test_predictions, test_true_labels = evaluate(model, test_loader, device)
test_precision = precision_score(test_true_labels, test_predictions, average='weighted')
test_recall = recall_score(test_true_labels, test_predictions, average='weighted')
test_f1 = f1_score(test_true_labels, test_predictions, average='weighted')

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1-Score: {test_f1:.4f}")

print("\nTest Set Classification Report:")
print(classification_report(test_true_labels, test_predictions))

# Summary comparison
print("\n" + "="*60)
print("SUMMARY: Validation vs Test Performance")
print("="*60)
print(f"{'Metric':<15} {'Validation':<15} {'Test':<15}")
print("-"*60)
print(f"{'Accuracy':<15} {val_accuracy:<15.4f} {test_accuracy:<15.4f}")
print(f"{'Precision':<15} {val_precision:<15.4f} {test_precision:<15.4f}")
print(f"{'Recall':<15} {val_recall:<15.4f} {test_recall:<15.4f}")
print(f"{'F1-Score':<15} {val_f1:<15.4f} {test_f1:<15.4f}")

In [ ]:
# Load best model for final evaluation
model.load_state_dict(torch.load('best_distilbert_model.pt'))

print("="*60)
print("FINAL MODEL EVALUATION")
print("="*60)

# Evaluate on validation set
print("\n--- Validation Set Evaluation ---")
val_loss, val_accuracy, val_predictions, val_true_labels = evaluate(model, val_loader, device)
val_precision = precision_score(val_true_labels, val_predictions, average='weighted')
val_recall = recall_score(val_true_labels, val_predictions, average='weighted')
val_f1 = f1_score(val_true_labels, val_predictions, average='weighted')

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Precision: {val_precision:.4f}")
print(f"Validation Recall: {val_recall:.4f}")
print(f"Validation F1-Score: {val_f1:.4f}")

print("\nValidation Set Classification Report:")
print(classification_report(val_true_labels, val_predictions))

# Evaluate on test set
print("\n--- Test Set Evaluation ---")
test_loss, test_accuracy, test_predictions, test_true_labels = evaluate(model, test_loader, device)
test_precision = precision_score(test_true_labels, test_predictions, average='weighted')
test_recall = recall_score(test_true_labels, test_predictions, average='weighted')
test_f1 = f1_score(test_true_labels, test_predictions, average='weighted')

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1-Score: {test_f1:.4f}")

print("\nTest Set Classification Report:")
print(classification_report(test_true_labels, test_predictions))

# Summary comparison
print("\n" + "="*60)
print("SUMMARY: Validation vs Test Performance")
print("="*60)
print(f"{'Metric':<15} {'Validation':<15} {'Test':<15}")
print("-"*60)
print(f"{'Accuracy':<15} {val_accuracy:<15.4f} {test_accuracy:<15.4f}")
print(f"{'Precision':<15} {val_precision:<15.4f} {test_precision:<15.4f}")
print(f"{'Recall':<15} {val_recall:<15.4f} {test_recall:<15.4f}")
print(f"{'F1-Score':<15} {val_f1:<15.4f} {test_f1:<15.4f}")

In [ ]:
# Load best model for final evaluation
model.load_state_dict(torch.load('best_distilbert_model.pt'))

print("="*60)
print("FINAL MODEL EVALUATION")
print("="*60)

# Evaluate on validation set
print("\n--- Validation Set Evaluation ---")
val_loss, val_accuracy, val_predictions, val_true_labels = evaluate(model, val_loader, device)
val_precision = precision_score(val_true_labels, val_predictions, average='weighted')
val_recall = recall_score(val_true_labels, val_predictions, average='weighted')
val_f1 = f1_score(val_true_labels, val_predictions, average='weighted')

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Precision: {val_precision:.4f}")
print(f"Validation Recall: {val_recall:.4f}")
print(f"Validation F1-Score: {val_f1:.4f}")

print("\nValidation Set Classification Report:")
print(classification_report(val_true_labels, val_predictions))

# Evaluate on test set
print("\n--- Test Set Evaluation ---")
test_loss, test_accuracy, test_predictions, test_true_labels = evaluate(model, test_loader, device)
test_precision = precision_score(test_true_labels, test_predictions, average='weighted')
test_recall = recall_score(test_true_labels, test_predictions, average='weighted')
test_f1 = f1_score(test_true_labels, test_predictions, average='weighted')

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test F1-Score: {test_f1:.4f}")

print("\nTest Set Classification Report:")
print(classification_report(test_true_labels, test_predictions))

# Summary comparison
print("\n" + "="*60)
print("SUMMARY: Validation vs Test Performance")
print("="*60)
print(f"{'Metric':<15} {'Validation':<15} {'Test':<15}")
print("-"*60)
print(f"{'Accuracy':<15} {val_accuracy:<15.4f} {test_accuracy:<15.4f}")
print(f"{'Precision':<15} {val_precision:<15.4f} {test_precision:<15.4f}")
print(f"{'Recall':<15} {val_recall:<15.4f} {test_recall:<15.4f}")
print(f"{'F1-Score':<15} {val_f1:<15.4f} {test_f1:<15.4f}")